## Exploracion del Dataset Crudo
En este notebook se realiza una exploración inicial del dataset crudo para entender su estructura, calidad y posibles problemas que puedan afectar el análisis posterior. Se incluyen pasos para la limpieza de datos, identificación de valores faltantes y análisis de distribuciones.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

# Configuración de Pandas para mostrar todas las columnas y filas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [7]:
# Carga y primera vista de los datos
df = pd.read_csv('../data/raw-data/pacientes.csv', sep=';')
print("Primeras filas del dataset:")
df.head()

Primeras filas del dataset:


,id_paciente,edad_paciente,sexo,peso_kg,altura_cm,fecha_registro,frecuencia_cardiaca_media_bpm,derivacion_ecg,frecuencia_muestreo_hz,etiqueta
0,P0305,82.0,M,69.8,168.0,4/10/2023,69.4,II,250,Normal
1,P0500,58.0,M,70.9,178.9,23/04/2023,79.2,II,250,Normal
2,P0442,49.0,M,84.2,173.1,25/01/2023,72.7,II,250,Normal
3,P0154,39.0,F,80.5,156.4,24/06/2023,87.0,II,250,Anormal
4,P0479,22.0,F,78.7,165.5,28/01/2023,77.8,II,250,Normal


In [8]:
# Información de los tipos de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 515 entries, 0 to 514
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_paciente                    515 non-null    str    
 1   edad_paciente                  511 non-null    float64
 2   sexo                           511 non-null    str    
 3   peso_kg                        509 non-null    float64
 4   altura_cm                      509 non-null    float64
 5   fecha_registro                 515 non-null    str    
 6   frecuencia_cardiaca_media_bpm  515 non-null    float64
 7   derivacion_ecg                 515 non-null    str    
 8   frecuencia_muestreo_hz         515 non-null    int64  
 9   etiqueta                       515 non-null    str    
dtypes: float64(4), int64(1), str(5)
memory usage: 40.4 KB


El dataset tiene **515 filas** y **10 columnas**: identificador, variables clínicas
(edad, sexo, peso, altura, fecha de registro), variables asociadas a la señal ECG
(frecuencia cardiaca media, derivación, frecuencia de muestreo) y la etiqueta objetivo.

### Tipos de datos

- Variables numéricas (`float64`): `edad_paciente`, `peso_kg`, `altura_cm`, `frecuencia_cardiaca_media_bpm`.
- Variable numérica entera (`int64`): `frecuencia_muestreo_hz`.
- Variables categóricas o de texto (`str`): `id_paciente`, `sexo`, `fecha_registro`, `derivacion_ecg` y `etiqueta`.

- La columna `fecha_registro` se encuentra almacenada como texto y será necesario verificar si requiere conversión al tipo `datetime`.


In [11]:
# Estadísticas descriptivas de las variables numéricas
df.describe()

,edad_paciente,peso_kg,altura_cm,frecuencia_cardiaca_media_bpm,frecuencia_muestreo_hz
count,511.000000,509.000000,509.000000,515.000000,515.0
mean,54.671233,72.233595,166.812377,80.997282,250.0
std,18.012891,11.579113,8.620153,17.229858,0.0
min,-5.000000,33.500000,143.800000,44.300000,250.0
25%,43.000000,64.600000,160.600000,68.050000,250.0
50%,56.000000,72.200000,166.200000,77.000000,250.0
75%,65.000000,80.300000,173.100000,90.850000,250.0
max,150.000000,102.600000,192.700000,142.300000,250.0


In [12]:
# Inspeccionar valores nulos en el dataset
df.isnull().sum()

id_paciente                      0
edad_paciente                    4
sexo                             4
peso_kg                          6
altura_cm                        6
fecha_registro                   0
frecuencia_cardiaca_media_bpm    0
derivacion_ecg                   0
frecuencia_muestreo_hz           0
etiqueta                         0
dtype: int64

In [15]:
# Identificar duplicados en la columna `id_paciente`
df["id_paciente"].duplicated().sum()

np.int64(15)

In [16]:
# Ver edades fuera de rango (0-120 años)
df["edad_paciente"].describe()

count    511.000000
mean      54.671233
std       18.012891
min       -5.000000
25%       43.000000
50%       56.000000
75%       65.000000
max      150.000000
Name: edad_paciente, dtype: float64

In [17]:
# Ver alturas fuera de rango (50-250 cm)
df["altura_cm"].describe()

count    509.000000
mean     166.812377
std        8.620153
min      143.800000
25%      160.600000
50%      166.200000
75%      173.100000
max      192.700000
Name: altura_cm, dtype: float64

In [18]:
# Ver fechas de registro para identificar su formato 
df["fecha_registro"].head(10)

0     4/10/2023
1    23/04/2023
2    25/01/2023
3    24/06/2023
4    28/01/2023
5    27/01/2023
6    29/06/2023
7     2/11/2023
8    21/02/2023
9    23/04/2023
Name: fecha_registro, dtype: str

## Valores faltantes

Se identificaron valores faltantes en cuatro variables:

edad_paciente: 4
sexo: 4
peso_kg: 6
altura_cm: 6

Las demás variables no presentan datos faltantes.

## Valores Duplicados

Se encontraron 15 filas duplicadas en la columna `id_paciente`, lo que indica que algunos pacientes tienen múltiples registros (2).

## Valores potencialmente inválidos

Durante la exploración se identificaron valores que requieren revisión:

- La edad mínima registrada es **-5 años**, un valor imposible desde el punto de vista clínico.
- La edad máxima registrada es **150 años**, valor que excede ampliamente el rango esperado para la población.

Estos registros serán analizados para decidir si deben corregirse, imputarse o eliminarse.



-----------------------------------------------